# Identifikasi Dataset Klasifikasi Biner (Binary Classification)

**Nama:** Mochamad Syauqil Huda  
**Kelas:** 05TPLP022  
**Mata Kuliah:** Machine Learning

---

## Tujuan Praktikum

Notebook ini bertujuan mengidentifikasi tiga dataset yang termasuk **klasifikasi biner**. Pemeriksaan dilakukan dengan membaca data, memeriksa kolom target, menghitung jumlah kelas, melihat data kosong, dan membuat grafik distribusi kelas.

### Pengertian klasifikasi biner

Klasifikasi biner adalah metode klasifikasi ketika setiap sampel hanya mempunyai **satu label target** dengan **tepat dua kemungkinan kelas**. Contohnya adalah prediksi suatu aplikasi termasuk kategori berbahaya atau tidak berbahaya.

### Kriteria pemeriksaan

1. Dataset mempunyai satu kolom target yang jelas.
2. Kolom target mempunyai tepat dua nilai kelas yang berbeda.
3. Setiap baris hanya mempunyai satu nilai target.

> Catatan: jumlah nilai pada kolom fitur tidak menentukan jenis klasifikasi. Jenis klasifikasi ditentukan berdasarkan kolom target.

## 1. Persiapan Library dan File Dataset

Notebook ini menggunakan file dataset yang sudah disediakan dalam file **`dataset.zip`**. Dataset dibaca secara lokal sehingga tidak perlu mengunduhnya lagi melalui `ucimlrepo`.

Letakkan file `dataset.zip` di folder yang sama dengan notebook ini. Kode di bawah akan mengekstrak file secara otomatis ke folder `dataset` jika belum tersedia.

Library yang digunakan:
- `pandas` untuk membaca dan memeriksa data.
- `matplotlib` untuk membuat grafik.
- `scipy` untuk membaca file Seismic Bumps berformat ARFF.

Jika library belum tersedia, jalankan cell instalasi. Setelah instalasi selesai, lakukan **Restart Kernel**, lalu jalankan notebook dari awal.

In [1]:
%pip install pandas matplotlib scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install --upgrade --force-reinstall --no-cache-dir --only-binary=:all: numpy scipy

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.6 MB 1.8 MB/s eta 0:00:07
   -- ------------------------------------- 0.8/12.6 MB 1.3 MB/s eta 0:00:09
   -- ------------------------------------- 0.8/12.6 MB 1.3 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.6 MB 1.2 MB/s eta 0:00:10
   ---- ----------------------------------- 1.3/12.6 MB 1.2 MB/s eta 0:00:10
   ----- ---------------------------------- 1.6/12.6 MB 1.2 MB/s eta 0:00:10
   ----- ---------------------------------- 1.6/12.6 MB 1.2 MB/s eta 0:00:10
   ----- ---------------------------------- 1.8/12.6 MB 1.0 MB/s eta 0:00:11
   ----- ---------------------------------- 1.8/12.6 MB 1.0 MB/s eta 0:00:11
   ------ --------------------------------- 2.1/12.6 MB 941.6 kB/s eta 0:00:12
   ------ --------------------------------- 2.1/12.6 MB 941.6 kB/s eta 0:00:12
   ------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import arff

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)

# Lokasi kerja notebook di VS Code
folder_kerja = Path.cwd()

# Cari folder dataset
kandidat_folder = [
    folder_kerja / "dataset",
    folder_kerja / "KLASIFIKASI BINER" / "dataset"
]

folder_dataset = next(
    (folder for folder in kandidat_folder if folder.is_dir()),
    None
)

if folder_dataset is None:
    raise FileNotFoundError(
        "Folder dataset tidak ditemukan. Periksa lokasi notebook dan folder dataset."
    )

# Daftar file dataset yang wajib tersedia
file_wajib = [
    folder_dataset / "TUANDROMD.csv",
    folder_dataset / "biodeg.csv",
    folder_dataset / "seismic-bumps.arff"
]

# Pastikan seluruh file tersedia
for file in file_wajib:
    if not file.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {file}")

print("Library berhasil diimpor.")
print("Folder kerja:", folder_kerja.resolve())
print("Folder dataset:", folder_dataset.resolve())
print("Semua file dataset ditemukan.")


Library berhasil diimpor.
Folder dataset: C:\Users\Syauqil\PRAKTIKUM MACHINE LEARNING\KLASIFIKASI BINER\dataset
Semua file dataset ditemukan.


## 2. Fungsi untuk Memeriksa Dataset

Fungsi berikut digunakan agar pemeriksaan ketiga dataset dilakukan dengan langkah yang sama. Informasi yang ditampilkan meliputi:
- ukuran dataset dan jumlah fitur;
- contoh data fitur dan target;
- jumlah nilai kosong;
- nama serta jumlah kelas target;
- distribusi jumlah data per kelas;
- grafik distribusi kelas;
- kesimpulan apakah dataset memenuhi kriteria klasifikasi biner.

Baris yang tidak mempunyai nilai target tidak digunakan dalam pemeriksaan kelas, karena baris tersebut tidak memiliki label yang dapat dipelajari.

In [ ]:
def periksa_dataset(X, y, nama_dataset, url_sumber):
    # Pastikan target berbentuk DataFrame dan memiliki satu kolom
    if isinstance(y, pd.Series):
        y = y.to_frame()

    print("=" * 75)
    print("NAMA DATASET:", nama_dataset)
    print("=" * 75)

    print("\\n1. Ukuran dataset")
    print("Jumlah baris/sampel :", X.shape[0])
    print("Jumlah fitur        :", X.shape[1])
    print("Jumlah kolom target :", y.shape[1])

    print("\\n2. Contoh fitur (5 baris pertama)")
    display(X.head())

    print("\\n3. Contoh target (5 baris pertama)")
    display(y.head())

    print("\\n4. Informasi nilai kosong")
    print("Nilai kosong pada fitur :", int(X.isna().sum().sum()))
    print("Nilai kosong pada target:", int(y.isna().sum().sum()))

    if y.shape[1] != 1:
        print("\\nPERINGATAN: Target bukan satu kolom. Perlu pemeriksaan lebih lanjut.")
        return X, y

    nama_target = y.columns[0]
    label = y[nama_target].astype("string").str.strip()

    # Baris dengan target kosong dikeluarkan dari penghitungan kelas
    valid = label.notna()
    label_valid = label[valid]
    distribusi = label_valid.value_counts()

    print("\\n5. Pemeriksaan label target")
    print("Nama kolom target       :", nama_target)
    print("Jumlah target kosong    :", int((~valid).sum()))
    print("Jumlah kelas unik       :", label_valid.nunique())
    print("Daftar kelas             :", label_valid.unique().tolist())

    print("\\nJumlah data per kelas:")
    display(distribusi.rename("Jumlah Data").to_frame())

    print("\\n6. Grafik distribusi kelas")
    ax = distribusi.plot(
        kind="bar",
        figsize=(7, 4),
        color="steelblue",
        edgecolor="black"
    )
    ax.set_title("Distribusi Label - " + nama_dataset)
    ax.set_xlabel("Kelas Target")
    ax.set_ylabel("Jumlah Data")
    ax.tick_params(axis="x", rotation=0)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    jumlah_kelas = label_valid.nunique()
    tepat_dua_kelas = jumlah_kelas == 2
    satu_kolom_target = y.shape[1] == 1

    print("\\n7. Hasil identifikasi")
    if satu_kolom_target and tepat_dua_kelas:
        print("VALID: target memiliki tepat dua kelas. Dataset termasuk klasifikasi biner.")
    else:
        print("PERLU DIPERIKSA: target belum menunjukkan tepat dua kelas.")

    if (~valid).sum() > 0:
        print("Catatan:", int((~valid).sum()),
              "baris dengan target kosong tidak dihitung dalam distribusi kelas.")

    print("\\nSumber:", url_sumber)
    return X, y

# Dataset 1 — TUANDROMD (Tezpur University Android Malware Dataset)

**Sumber:** UCI Machine Learning Repository  
**URL:** https://archive.ics.uci.edu/dataset/855/tuandromd  
**File lokal:** `dataset/TUANDROMD.csv`

### Deskripsi dataset

TUANDROMD berisi karakteristik aplikasi Android. Fitur-fiturnya mencakup izin (*permissions*) aplikasi dan pemanggilan API. Dataset ini dapat digunakan untuk membedakan aplikasi berdasarkan label target yang tersedia.

Pada file yang digunakan di praktikum ini, kolom target bernama **`Label`** dan memiliki dua nilai, yaitu **0** dan **1**. Nilai tersebut dipertahankan sesuai pengkodean pada file sumber agar tidak terjadi kesalahan pemetaan label.

### Alasan dugaan klasifikasi

Dataset ini diduga termasuk klasifikasi biner karena setiap aplikasi mempunyai satu nilai pada kolom target `Label`, dan target tersebut memiliki dua kemungkinan nilai. Pemeriksaan jumlah kelas dilakukan menggunakan Python.

In [ ]:
# Membaca dataset TUANDROMD dari file CSV lokal
file_tuan = folder_dataset / "TUANDROMD.csv"
df_tuan = pd.read_csv(file_tuan)

print("Ukuran data mentah:", df_tuan.shape)
print("Nama kolom target: Label")
print("Distribusi label mentah:")
display(df_tuan["Label"].value_counts(dropna=False).rename("Jumlah Data").to_frame())

# Memisahkan fitur dan target
X_tuan = df_tuan.drop(columns=["Label"])
y_tuan = df_tuan[["Label"]]

X_tuan, y_tuan = periksa_dataset(
    X=X_tuan,
    y=y_tuan,
    nama_dataset="TUANDROMD",
    url_sumber="https://archive.ics.uci.edu/dataset/855/tuandromd"
)

### Analisis dan alasan klasifikasi TUANDROMD

Kolom target pada dataset TUANDROMD adalah `Label`. Pada file yang digunakan, target mempunyai dua nilai kelas, yaitu **0 dan 1**. Terdapat satu baris dengan target kosong; baris tersebut tidak dihitung sebagai kelas karena tidak mempunyai label.

Dataset ini termasuk **klasifikasi biner** karena setiap aplikasi hanya memiliki satu target dan target tersebut memiliki dua kemungkinan kelas. Banyaknya fitur izin dan API tidak mengubah jenis klasifikasi, karena penentuan jenis klasifikasi dilihat dari kolom target.

# Dataset 2 — QSAR Biodegradation

**Sumber:** UCI Machine Learning Repository  
**URL:** https://archive.ics.uci.edu/dataset/254/qsar+biodegradation  
**File lokal:** `dataset/biodeg.csv`

### Deskripsi dataset

QSAR Biodegradation berisi data senyawa kimia dengan sejumlah *molecular descriptors* atau karakteristik molekul. Dataset ini digunakan untuk mengelompokkan senyawa berdasarkan kemampuan biodegradasinya.

Dua label target yang digunakan:
- **RB (Ready Biodegradable):** senyawa siap atau mudah terurai secara biologis.
- **NRB (Not Ready Biodegradable):** senyawa tidak termasuk kategori siap terurai secara biologis.

### Alasan dugaan klasifikasi

Dataset ini diduga termasuk klasifikasi biner karena setiap senyawa memiliki satu label target, yaitu RB atau NRB. File CSV ini tidak memiliki baris header, sehingga kolom fitur dan target perlu diberi nama saat dibaca.

In [ ]:
# Membaca QSAR Biodegradation
# File menggunakan pemisah titik koma (;) dan tidak memiliki header.
file_qsar = folder_dataset / "biodeg.csv"

nama_fitur_qsar = [f"fitur_{i:02d}" for i in range(1, 42)]
nama_kolom_qsar = nama_fitur_qsar + ["Class"]

df_qsar = pd.read_csv(
    file_qsar,
    sep=";",
    header=None,
    names=nama_kolom_qsar
)

print("Ukuran data:", df_qsar.shape)
print("Distribusi label:")
display(df_qsar["Class"].value_counts().rename("Jumlah Data").to_frame())

X_qsar = df_qsar.drop(columns=["Class"])
y_qsar = df_qsar[["Class"]]

X_qsar, y_qsar = periksa_dataset(
    X=X_qsar,
    y=y_qsar,
    nama_dataset="QSAR Biodegradation",
    url_sumber="https://archive.ics.uci.edu/dataset/254/qsar+biodegradation"
)

### Analisis dan alasan klasifikasi QSAR Biodegradation

Kolom target dataset ini bernama `Class` dan mempunyai dua kategori, yaitu **RB** dan **NRB**. Setiap sampel senyawa kimia hanya memiliki satu label target.

Oleh karena itu, QSAR Biodegradation termasuk **klasifikasi biner**. Model hanya perlu menentukan apakah suatu senyawa termasuk kategori siap terurai secara biologis atau tidak.

# Dataset 3 — Seismic Bumps

**Sumber:** UCI Machine Learning Repository  
**URL:** https://archive.ics.uci.edu/dataset/266/seismic+bumps  
**File lokal:** `dataset/seismic-bumps.arff`

### Deskripsi dataset

Seismic Bumps berisi data aktivitas seismik di area pertambangan batu bara. Dataset ini digunakan untuk memperkirakan apakah pada giliran kerja berikutnya akan terjadi *seismic bump* berenergi tinggi.

Kolom target bernama **`class`**, dengan dua nilai:
- **1:** terjadi *seismic bump* berenergi tinggi.
- **0:** tidak terjadi *seismic bump* berenergi tinggi.

### Alasan dugaan klasifikasi

Dataset ini diduga termasuk klasifikasi biner karena target `class` hanya memiliki dua kemungkinan nilai, yaitu 0 dan 1. File dataset menggunakan format ARFF sehingga dibaca menggunakan fungsi dari library SciPy.

In [ ]:
# Membaca Seismic Bumps dari file ARFF
file_seismic = folder_dataset / "seismic-bumps.arff"

data_arff, metadata_arff = arff.loadarff(file_seismic)
df_seismic = pd.DataFrame(data_arff)

# Mengubah nilai bytes menjadi teks agar label lebih mudah dibaca
for kolom in df_seismic.select_dtypes(include=["object"]).columns:
    df_seismic[kolom] = df_seismic[kolom].apply(
        lambda nilai: nilai.decode("utf-8") if isinstance(nilai, bytes) else nilai
    )

print("Ukuran data:", df_seismic.shape)
print("Distribusi label:")
display(df_seismic["class"].value_counts().rename("Jumlah Data").to_frame())

X_seismic = df_seismic.drop(columns=["class"])
y_seismic = df_seismic[["class"]]

X_seismic, y_seismic = periksa_dataset(
    X=X_seismic,
    y=y_seismic,
    nama_dataset="Seismic Bumps",
    url_sumber="https://archive.ics.uci.edu/dataset/266/seismic+bumps"
)

### Analisis dan alasan klasifikasi Seismic Bumps

Kolom target pada Seismic Bumps adalah `class`, dengan dua nilai yaitu **0** dan **1**. Dengan demikian, dataset ini memenuhi kriteria **klasifikasi biner** karena setiap sampel hanya mempunyai satu target dengan dua kemungkinan kelas.

Perlu diperhatikan bahwa jumlah kelas pada dataset ini tidak seimbang. Jumlah data dengan label 0 jauh lebih banyak daripada label 1. Kondisi ini disebut *class imbalance* dan perlu diperhatikan apabila dataset digunakan untuk membangun model klasifikasi.

# Kesimpulan

Berdasarkan pemeriksaan terhadap ketiga dataset, jenis klasifikasi ditentukan dari kolom target, bukan dari banyaknya fitur.

1. **TUANDROMD:** mempunyai satu kolom target `Label` dengan dua nilai kelas, yaitu 0 dan 1. Satu baris dengan target kosong tidak dihitung dalam distribusi kelas.
2. **QSAR Biodegradation:** mempunyai satu kolom target `Class` dengan dua kategori, yaitu RB dan NRB.
3. **Seismic Bumps:** mempunyai satu kolom target `class` dengan dua nilai, yaitu 0 dan 1. Dataset ini memiliki distribusi kelas yang tidak seimbang.

Ketiga dataset tersebut memenuhi kriteria **klasifikasi biner** karena masing-masing memiliki satu target dengan tepat dua kelas yang dapat diprediksi.

---

## Referensi

1. UCI Machine Learning Repository. TUANDROMD: https://archive.ics.uci.edu/dataset/855/tuandromd
2. UCI Machine Learning Repository. QSAR Biodegradation: https://archive.ics.uci.edu/dataset/254/qsar+biodegradation
3. UCI Machine Learning Repository. Seismic Bumps: https://archive.ics.uci.edu/dataset/266/seismic+bumps